In [25]:
import os
import json
import pandas as pd
import re
import numpy as np


# Connect to Google Drive
import gspread
import gspread_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from googleapiclient.discovery import build
from gspread_dataframe import set_with_dataframe
from gspread_dataframe import get_as_dataframe

In [ ]:
# 1. Fetch credentials from environment variable
creds_env = os.environ.get("GDRIVE_CREDENTIALS_KC")

if not creds_env:
    raise ValueError("Environment variable 'GDRIVE_CREDENTIALS' was not found.")

creds_json = json.loads(creds_env)

# 2. Define required scopes
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# 3. Authenticate service account
creds = service_account.Credentials.from_service_account_info(
    creds_json, scopes=scopes
)

# 4. Initialize Google API clients
drive_service = build("drive", "v3", credentials=creds)
sheets_service = build("sheets", "v4", credentials=creds)

gc = gspread.authorize(creds)

print("Google Drive and Sheets services successfully initialized.")

Mounted at /content/drive


In [28]:
# Open files

forms_data = gc.open_by_key('1xmm88zF84MEk-7ZwARh6ZCFcaI2SmW8c7Dnlwe4zGr4')
forms_data = forms_data.get_worksheet(0)
forms_data = get_as_dataframe(forms_data)

influencers_posts = gc.open_by_key('1ckdCSF1JD_SMkBNPuLfb0FdsuHQ9wXkL4u2NoLWe8aQ')
influencers_posts = influencers_posts.get_worksheet(0)
influencers_posts = get_as_dataframe(influencers_posts)

influencers_comments = gc.open_by_key('1a592d89m2sCmcxuKqF2SeJRGFUYanZIXo0RKOxmVVpU')
influencers_comments = influencers_comments.get_worksheet(0)
influencers_comments = get_as_dataframe(influencers_comments)

posts_data = gc.open_by_key('1GWe4XOZatmvNknH6GmvrEKnDjNSTUK4OOQoqV4So0zQ')
posts_data = posts_data.worksheet('tt_data_post_post_max')
posts_data = get_as_dataframe(posts_data)

posts_comments = gc.open_by_key('1nHmuARdf6vLGGZw2ykXlL5zJ_KOa15-7BARxGHaGP2U')
posts_comments = posts_comments.get_worksheet(0)
posts_comments = get_as_dataframe(posts_comments)

In [29]:
# Clean databases
forms_data = forms_data.fillna(0)
influencers_posts = influencers_posts.fillna(0)
influencers_comments = influencers_comments.fillna(0)
posts_data = posts_data.fillna(0)
posts_comments = posts_comments.fillna(0)

/tmp/ipykernel_2580/135062332.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  posts_comments = posts_comments.fillna(0)


In [30]:
# Consolidate sentiment for each published post
posts_comments = posts_comments.rename(columns={'username': 'commenter'})
posts_comments = posts_comments.merge(posts_data[["video_url", "username"]], on='video_url', how='left')
sentiment_posts = posts_comments.groupby(['video_url', 'username']).agg(
    positive_count=('classification', lambda x: x.isin(['promotor', 'Promoter', 'promoter', 'Promotor']).sum()),
    negative_count=('classification', lambda x: x.isin(['detrator', 'Detractor', 'detractor', 'Detractor']).sum()),
    neutral_count=('classification', lambda x: x.isin(['neutro', 'neutral', 'Neutro', 'Neutral']).sum()),
).reset_index()
# 2. Sum the three sentiment columns to get the total
sentiment_posts['total_sentiments'] = (
    sentiment_posts['positive_count'] +
    sentiment_posts['negative_count'] +
    sentiment_posts['neutral_count']
)

# 3. Calculate the percentage of each sentiment over the total
sentiment_posts['positive_percentage'] = sentiment_posts['positive_count'] / sentiment_posts['total_sentiments']
sentiment_posts['negative_percentage'] = sentiment_posts['negative_count'] / sentiment_posts['total_sentiments']
sentiment_posts['neutral_percentage'] = sentiment_posts['neutral_count'] / sentiment_posts['total_sentiments']

In [31]:
sentiment_posts

,video_url,username,positive_count,negative_count,neutral_count,total_sentiments,positive_percentage,negative_percentage,neutral_percentage
0,https://www.tiktok.com/@_dulcepink_/video/7608...,_dulcepink_,24,15,12,51,0.470588,0.294118,0.235294
1,https://www.tiktok.com/@_dulcepink_/video/7612...,_dulcepink_,35,50,15,100,0.350000,0.500000,0.150000
2,https://www.tiktok.com/@benjacaldero/video/760...,benjacaldero,36,16,48,100,0.360000,0.160000,0.480000
3,https://www.tiktok.com/@benjacaldero/video/761...,benjacaldero,32,36,32,100,0.320000,0.360000,0.320000
4,https://www.tiktok.com/@caam.prz/video/7608226...,caam.prz,2,3,2,7,0.285714,0.428571,0.285714
5,https://www.tiktok.com/@caam.prz/video/7609874...,caam.prz,32,43,25,100,0.320000,0.430000,0.250000
6,https://www.tiktok.com/@crispierri/video/76083...,crispierri,3,4,13,20,0.150000,0.200000,0.650000
7,https://www.tiktok.com/@davooxeneize/video/765...,davooxeneize,27,58,15,100,0.270000,0.580000,0.150000
8,https://www.tiktok.com/@davooxeneize/video/765...,davooxeneize,35,48,17,100,0.350000,0.480000,0.170000
9,https://www.tiktok.com/@florjazminp/video/7642...,florjazminp,47,43,10,100,0.470000,0.430000,0.100000


In [32]:
# Prepare datasets for join

posts_data = posts_data.rename(columns={'username': 'influencer'})
posts_data = posts_data.rename(columns={'video_url': 'url'})
forms_data = forms_data.rename(columns={'Link of Post': 'url'})
sentiment_posts = sentiment_posts.rename(columns={'video_url': 'url'})
influencers_posts_join = (
    influencers_posts[["username", "followers"]]
    .drop_duplicates(subset="username")
    .rename(columns={"username": "influencer"})
)


In [33]:
# Join datasets
tiktok_influencers_final = posts_data
tiktok_influencers_final = tiktok_influencers_final.merge(forms_data, on='url', how='left')
tiktok_influencers_final = tiktok_influencers_final.merge(sentiment_posts, on='url', how='left')
tiktok_influencers_final = tiktok_influencers_final.merge(influencers_posts_join, on='influencer', how='left')

tiktok_influencers_final

,url,influencer,run_datetime,aweme_id,likes,comment_count,share_count,views,saves,download_count,...,Budget: AON / campaña,username,positive_count,negative_count,neutral_count,total_sentiments,positive_percentage,negative_percentage,neutral_percentage,followers
0,https://www.tiktok.com/@robergalati/video/7641...,robergalati,2026-08-26 12:49:11,7.641638e+18,32936.0,113.0,1791.0,24243357.0,3018.0,51.0,...,0.0,robergalati,29.0,42.0,20.0,91.0,0.318681,0.461538,0.219780,37265.0
1,https://www.tiktok.com/@florjazminp/video/7642...,florjazminp,2026-08-26 12:50:12,7.642103e+18,49120.0,456.0,531.0,5456485.0,1564.0,9.0,...,0.0,florjazminp,47.0,43.0,10.0,100.0,0.470000,0.430000,0.100000,1814675.0
2,https://www.tiktok.com/@gastonedul/video/76425...,gastonedul,2026-08-26 12:51:07,7.642503e+18,25920.0,251.0,503.0,6164281.0,1144.0,6.0,...,0.0,gastonedul,27.0,49.0,24.0,100.0,0.270000,0.490000,0.240000,2111301.0
3,https://www.tiktok.com/@gastonedul/video/76524...,gastonedul,2026-08-26 12:52:01,7.652461e+18,8390.0,94.0,22.0,80064.0,225.0,5.0,...,0.0,gastonedul,24.0,15.0,20.0,59.0,0.406780,0.254237,0.338983,2111301.0
4,https://www.tiktok.com/@laagusneta/video/76428...,laagusneta,2026-08-26 12:52:41,7.642804e+18,25873.0,137.0,1126.0,9196369.0,1382.0,8.0,...,0.0,laagusneta,22.0,32.0,26.0,80.0,0.275000,0.400000,0.325000,551389.0
5,https://www.tiktok.com/@owenn_27/video/7645441...,owenn_27,2026-08-26 12:53:30,7.645441e+18,5747.0,40.0,24.0,113871.0,187.0,6.0,...,0.0,owenn_27,12.0,5.0,9.0,26.0,0.461538,0.192308,0.346154,5826203.0
6,https://www.tiktok.com/@owenn_27/video/7653281...,owenn_27,2026-08-26 12:53:55,7.653281e+18,2902.0,34.0,36.0,51360.0,129.0,3.0,...,0.0,owenn_27,16.0,7.0,7.0,30.0,0.533333,0.233333,0.233333,5826203.0
7,https://www.tiktok.com/@lulignzalez/video/7644...,lulignzalez,2026-08-26 12:54:15,7.644266e+18,12983.0,53.0,62.0,115636.0,194.0,12.0,...,0.0,lulignzalez,29.0,9.0,7.0,45.0,0.644444,0.200000,0.155556,1569717.0
8,https://www.tiktok.com/@lulignzalez/video/7656...,lulignzalez,2026-08-26 12:54:50,7.656832e+18,7585.0,46.0,48.0,77667.0,121.0,3.0,...,0.0,lulignzalez,24.0,6.0,6.0,36.0,0.666667,0.166667,0.166667,1569717.0
9,https://www.tiktok.com/@lacobraaa.9/video/7650...,lacobraaa.9,2026-08-26 12:55:20,7.650666e+18,60006.0,795.0,1407.0,8142117.0,2115.0,13.0,...,0.0,lacobraaa.9,13.0,64.0,23.0,100.0,0.130000,0.640000,0.230000,4382721.0


In [34]:
# Create Organic_id
tiktok_influencers_final["Organic_ID"] = tiktok_influencers_final["aweme_id"]
tiktok_influencers_final['Organic_ID'] = tiktok_influencers_final['Organic_ID'].astype('string')

In [35]:
# Rename columns
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'desc': 'copy'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'Published date': 'date_published'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'Plataform': 'platform'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'comment_count': 'comments'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'share_count': 'shares'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'Marca': 'brand'})


In [36]:
# Crear columnas

tiktok_influencers_final["content_type"] = "Influencers"
tiktok_influencers_final["format"] = "Video"
tiktok_influencers_final["total_interactions"] = tiktok_influencers_final["likes"] + tiktok_influencers_final["comments"] + tiktok_influencers_final["shares"] + tiktok_influencers_final["saves"]
tiktok_influencers_final["engagement_rate"] = (
    tiktok_influencers_final["total_interactions"]
    .div(tiktok_influencers_final["views"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
tiktok_influencers_final["positive_comments"] = tiktok_influencers_final["positive_percentage"] * tiktok_influencers_final["comments"]
tiktok_influencers_final["negative_comments"] = tiktok_influencers_final["negative_percentage"] * tiktok_influencers_final["comments"]
tiktok_influencers_final["neutral_comments"] = tiktok_influencers_final["neutral_percentage"] * tiktok_influencers_final["comments"]




In [37]:
# Seleccionar y ordenar columnas

cols = [
    "url",
    "copy",
    "date_published",
    "platform",
    "format",
    "influencer",
    "Country",
    "Organic_ID",
    "brand",
    "content_type",
    "views",
    "likes",
    "comments",
    "shares",
    "saves",
    "total_interactions",
    "engagement_rate",
    "positive_percentage",
    "negative_percentage",
    "neutral_percentage",
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "run_datetime",
    "followers"
]

tiktok_influencers_final = tiktok_influencers_final[cols]
tiktok_influencers_final = tiktok_influencers_final.reindex(columns=cols)

In [38]:
# Adjust date
tiktok_influencers_final["date_published"] = pd.to_datetime(
    tiktok_influencers_final["date_published"],
    format="mixed",
    errors="coerce",
).dt.date

In [40]:
# Save final table
# Open the destination sheets file
sh = gc.open_by_key('1FAVS_dtKl5WPoXeqpa5JGPbSe7tVv7GhumQvkUOSxiw')
worksheet = sh.get_worksheet(0)

# Replace old data with new data
set_with_dataframe(worksheet, tiktok_influencers_final)
print("DataFrame saved successfully!")

DataFrame saved successfully!
